In [1]:

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold

In [7]:

# ============================================================
# PATHS
# ============================================================

RAW_DATA_PATH = "../data/raw/retail_customers_COMPLETE_CATEGORICAL.csv"
TRAIN_TEST_PATH = "data/train_test/"
PROCESSED_PATH = "data/processed/"
MODELS_PATH = "models/"

os.makedirs(TRAIN_TEST_PATH, exist_ok=True)
os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)

# ============================================================
# LOAD DATA
# ============================================================

def load_data(path):
    df = pd.read_csv(path)
    print("Dataset loaded:", df.shape)
    print("Columns:", df.columns.tolist())
    return df

# ============================================================
# FEATURE ENGINEERING
# ============================================================

def feature_engineering(df):

    # -------------------------
    # Date parsing
    # -------------------------
    if "RegistrationDate" in df.columns:
        df["RegistrationDate"] = pd.to_datetime(
            df["RegistrationDate"],
            dayfirst=True,
            errors="coerce"
        )
        df["RegYear"] = df["RegistrationDate"].dt.year
        df["RegMonth"] = df["RegistrationDate"].dt.month
        df["RegDay"] = df["RegistrationDate"].dt.day
        df["RegWeekday"] = df["RegistrationDate"].dt.weekday
        df.drop(columns=["RegistrationDate"], inplace=True)

    # -------------------------
    # Business ratios
    # -------------------------
    if {"MonetaryTotal", "Recency"}.issubset(df.columns):
        df["MonetaryPerDay"] = df["MonetaryTotal"] / (df["Recency"] + 1)
    if {"MonetaryTotal", "Frequency"}.issubset(df.columns):
        df["AvgBasketValue"] = df["MonetaryTotal"] / (df["Frequency"] + 1)
    if {"Recency", "CustomerTenureDays"}.issubset(df.columns):
        df["TenureRatio"] = df["Recency"] / (df["CustomerTenureDays"] + 1)

    # -------------------------
    # IP feature
    # -------------------------
    if "LastLoginIP" in df.columns:
        df["IsPrivateIP"] = df["LastLoginIP"].astype(str).apply(
            lambda x: 1 if x.startswith(("10.", "192.168.", "172.")) else 0
        )
        df.drop(columns=["LastLoginIP"], inplace=True)

    # -------------------------
    # Drop useless / constant features
    # -------------------------
    if "NewsletterSubscribed" in df.columns:
        df.drop(columns=["NewsletterSubscribed"], inplace=True)

    return df

# ============================================================
# REMOVE MULTICOLLINEARITY
# ============================================================

def remove_multicollinearity(df, threshold=0.85):
    numeric_df = df.select_dtypes(include=["int64", "float64"])
    corr_matrix = numeric_df.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    print("Dropping correlated features:", to_drop)
    df.drop(columns=to_drop, inplace=True)
    return df

# ============================================================
# MAIN PREPROCESSING FUNCTION
# ============================================================

def preprocess():

    # 1️⃣ Load
    df = load_data(RAW_DATA_PATH)

    # 2️⃣ Target separation
    y = df["Churn"]
    X = df.drop(columns=["Churn"])

    # 3️⃣ Feature engineering
    X = feature_engineering(X)

    # 4️⃣ Remove highly correlated features
    DROP_HIGH_CORR = [
        "MonetaryAvg",
        "MonetaryStd",
        "MonetaryMax",
        "AvgQuantityPerTransaction",
        "MaxQuantity",
        "UniqueInvoices"
    ]
    X.drop(columns=[c for c in DROP_HIGH_CORR if c in X.columns], inplace=True)
    X = remove_multicollinearity(X, threshold=0.85)

    # 5️⃣ Identify column types
    numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
    categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

    print("Numerical cols:", numerical_cols)
    print("Categorical cols:", categorical_cols)

    # 6️⃣ Pipelines
    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer([
        ("num", numerical_pipeline, numerical_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ])

    # 7️⃣ Train/Test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # 8️⃣ Fit transform on train only
    X_train_proc = preprocessor.fit_transform(X_train)
    X_test_proc = preprocessor.transform(X_test)

    # 9️⃣ Optional PCA
    pca = PCA(n_components=0.95, random_state=42)
    X_train_pca = pca.fit_transform(X_train_proc)
    X_test_pca = pca.transform(X_test_proc)

    print("Train shape:", X_train_pca.shape)
    print("Test shape:", X_test_pca.shape)

    # 10️⃣ Save processed datasets
    pd.DataFrame(X_train_pca).to_csv(TRAIN_TEST_PATH + "X_train.csv", index=False)
    pd.DataFrame(X_test_pca).to_csv(TRAIN_TEST_PATH + "X_test.csv", index=False)
    y_train.to_csv(TRAIN_TEST_PATH + "y_train.csv", index=False)
    y_test.to_csv(TRAIN_TEST_PATH + "y_test.csv", index=False)

    # 11️⃣ Save preprocessing pipeline & PCA
    joblib.dump(preprocessor, MODELS_PATH + "preprocessor.pkl")
    joblib.dump(pca, MODELS_PATH + "pca.pkl")

    print("✅ Preprocessing completed successfully!")

# ============================================================
# EXECUTE
# ============================================================

if __name__ == "__main__":
    preprocess()

Dataset loaded: (4372, 52)
Columns: ['CustomerID', 'Recency', 'Frequency', 'MonetaryTotal', 'MonetaryAvg', 'MonetaryStd', 'MonetaryMin', 'MonetaryMax', 'TotalQuantity', 'AvgQuantityPerTransaction', 'MinQuantity', 'MaxQuantity', 'CustomerTenureDays', 'FirstPurchaseDaysAgo', 'PreferredDayOfWeek', 'PreferredHour', 'PreferredMonth', 'WeekendPurchaseRatio', 'AvgDaysBetweenPurchases', 'UniqueProducts', 'UniqueDescriptions', 'AvgProductsPerTransaction', 'UniqueCountries', 'NegativeQuantityCount', 'ZeroPriceCount', 'CancelledTransactions', 'ReturnRatio', 'TotalTransactions', 'UniqueInvoices', 'AvgLinesPerInvoice', 'Age', 'RegistrationDate', 'NewsletterSubscribed', 'LastLoginIP', 'SupportTicketsCount', 'SatisfactionScore', 'RFMSegment', 'AgeCategory', 'SpendingCategory', 'CustomerType', 'FavoriteSeason', 'PreferredTimeOfDay', 'Region', 'LoyaltyLevel', 'ChurnRiskCategory', 'WeekendPreference', 'BasketSizeCategory', 'ProductDiversity', 'Gender', 'AccountStatus', 'Country', 'Churn']
Dropping corre

C:\Users\ASUS\AppData\Local\Temp\ipykernel_17416\1669187929.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["RegistrationDate"] = pd.to_datetime(
C:\Users\ASUS\AppData\Local\Temp\ipykernel_17416\1669187929.py:115: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()


Train shape: (3497, 39)
Test shape: (875, 39)
✅ Preprocessing completed successfully!


Churn
0    0.667429
1    0.332571
Name: proportion, dtype: float64
